# 07 · SEI 상 매핑 (범용·투명) — 있나 / 얼마나 / 어디에

5개 후보상(LiF·Li2O·Li3N·Li2CO3·Li2S)을 **NBD(halo·ring·spot) + cepstral**로 스크리닝. 엔진 함수는 패키지에
있어 **범용**이고, 이 노트북은 **각 단계 셀 맨 위에 파라미터를 노출**해서 데이터마다 직접 조절한다.
기본값은 robust(median+MAD, 물리 q-창)라 그대로도 5~6개 데이터에 돈다 — **필요할 때만 그 셀 값을 바꾸면 됨.**

판정: **confirmed**=고유 링/스팟 보유 / **possible**=일관하나 고유 없음 / **weak/absent**=강한 링 없음.

## 0) 설정 & 로드 (경로만 바꾸면 어느 데이터든)

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
import fourdstem as fds

DM4_PATH   = "/home/jonghoonk918/Desktop/fdstem/Amorphous/Li-SEI/P-Cu/P-Gu1.dm4"   # ★데이터셋
USE_SYNTHETIC = not os.path.isfile(DM4_PATH)
DET_BIN    = 1                       # 검출기 비닝(메모리). q_max는 안 변함
Q_UNIT_HINT= "1/nm"                 # dm 단위(0.043888 1/nm)
CANDIDATES = ["LiF","Li2O","Li3N","Li2CO3","Li2S"]
VCOL = {"confirmed":"#2ca02c","possible":"#ff7f0e","weak/absent":"#7f7f7f"}

def _synth(Sy=22,Sx=30,H=88,W=88,seed=0):
    rng=np.random.default_rng(seed); yy,xx=np.mgrid[0:H,0:W]; cx,cy=W/2,H/2; rr=np.hypot(xx-cx,yy-cy)
    beam=8*np.exp(-rr**2/8); halo=lambda r0,s=4:np.exp(-(rr-r0)**2/(2*s**2))
    def sp(r0,n=6,a=4):
        im=np.zeros((H,W))
        for k in range(n):
            t=2*np.pi*k/n; im+=a*np.exp(-((xx-cx-r0*np.cos(t))**2+(yy-cy-r0*np.sin(t))**2)/(2*1.6**2))
        return im
    cube=np.empty((Sy,Sx,H,W),np.float32)
    for iy in range(Sy):
        for ix in range(Sx):
            base=0.9*beam if iy>=Sy-3 else (beam+sp(20)+0.5*halo(20) if ix<Sx//2 else beam+1.2*halo(18))
            cube[iy,ix]=np.clip(base+0.15*rng.standard_normal((H,W)),0,None)
    return cube
if USE_SYNTHETIC: cube=fds.from_array(_synth(),q_per_px=0.02,name="synthetic")
else:
    cube=fds.load(DM4_PATH,Q_UNIT_HINT)
    if DET_BIN>1: cube=fds.bin_cube_detector(cube,DET_BIN)
scan=cube.scan_shape; QPP=cube.calibration.q_per_px
NAME=("synthetic" if USE_SYNTHETIC else os.path.splitext(os.path.basename(DM4_PATH))[0])
SAVE_DIR=("nb7_outputs" if USE_SYNTHETIC else os.path.dirname(DM4_PATH)+"/nb7_outputs"); os.makedirs(SAVE_DIR,exist_ok=True)
def save(fig,n): p=os.path.join(SAVE_DIR,f"{NAME}_{n}.png"); fig.savefig(p,dpi=150,bbox_inches="tight"); print("saved:",p)
def save_csv(n,h,rows):
    import csv; p=os.path.join(SAVE_DIR,f"{NAME}_{n}.csv")
    with open(p,"w",newline="") as f: w=csv.writer(f); w.writerow(h); w.writerows(rows)
    print("saved:",p)
print("cube:",cube.shape,"| q_per_px=",QPP,"->",os.path.abspath(SAVE_DIR))

## 0.5) 전처리 진단 — **측정 먼저, 필요할 때만 교정** (과잉처리로 약신호 버리지 않게)

In [ ]:
# --- 이 셀 파라미터 ---
HOT_THRESHOLD   = 8.0    # hot/dead 검출 민감도(작을수록 민감). ★진짜 스팟이 지워지면 값을 키우세요
WANDER_WARN_PX  = 1.0    # 이보다 크면 per-position 정렬 권고
ELLIP_WARN      = 0.02   # 타원율 이보다 크면 타원 보정 권고
diag = fds.diagnose_cube(cube, hot_threshold=HOT_THRESHOLD)
center = diag["center"]
print("=== 전처리 진단 ===")
print(f"  center           = ({center[0]:.1f},{center[1]:.1f})   (hot-pixel 제거 후 무게중심)")
print(f"  beam wander      = {diag['wander_px']:.2f} px   (>{WANDER_WARN_PX} 이면 정렬 고려)")
print(f"  detector defects = {100*diag['bad_pixel_frac']:.2f} %  (defect-map만 안전 수리)")
print(f"  ring ellipticity = {100*diag['ellipticity']:.1f} % @ {diag['ellipse_angle_deg']:.0f}deg  (>{100*ELLIP_WARN:.0f}% 면 보정)")
for n in diag["notes"]: print("  -",n)
print("원칙: 인공물(타원/wander/defect/배경)은 교정 O, 신호 평활(블러/공격필터)은 X.")

## 1) 중심·패턴·물질 마스크 — 개요

물질 마스크는 **빔감쇠 t/lambda = ln(I_total/I_beam)** 로 잡는다. 얇은 물질은 링 산란이 약해 annular(밝기)로는
진공과 안 갈리지만, 빔감쇠는 **얇은 물질도 진공과 확실히 구분**한다. 기준 진공은 '총세기 하위 %'(확실히 빈 곳)로
엄격히 잡아 오염을 없앤다. **검은 영역이 과하게 잡히면 `MAT_TL_THRESH`를 낮춰라.**

In [ ]:
# --- 이 셀 파라미터 ---
CENTER        = None      # None=진단값(자동). 수동이면 (cx,cy)
BEAM_RADIUS_PX= None      # 직접빔 디스크 반경(px). None=자동(~det/20)
VAC_PCTL      = 15        # 총세기 하위 X% = '기준 진공'(확실히 빈 곳). 물질이 화면을 많이 채우면 낮추기
MAT_TL_THRESH = 0.15      # t/lambda 이보다 크면 '물질'. ★검은 영역이 과하면(얇은 물질까지 버림) 낮추기
if CENTER is not None: center=CENTER
med=fds.median_pattern(cube); mx=fds.clean_pattern(np.asarray(cube.max_dp(),float),hot_threshold=HOT_THRESHOLD)
# 물질 마스크: 얇은 물질은 링 산란이 약해 annular로는 진공과 안 갈림 → 빔감쇠 t/lambda 로 판정(얇은것도 잡힘)
_flat=cube._flat_patterns(); _tot=np.asarray(_flat,float).reshape(_flat.shape[0],-1).sum(1)
vac_ref=np.asarray(_tot<=np.percentile(_tot,VAC_PCTL),bool).reshape(scan)      # 기준 진공(엄격)
tmap,tex=fds.thickness_map(cube,center=center,beam_radius=BEAM_RADIUS_PX,vacuum_mask=vac_ref,return_extras=True)
_vn=1.4826*np.median(np.abs(tmap[vac_ref]-np.median(tmap[vac_ref])))           # 진공 노이즈(MAD)
material=np.asarray(tmap>MAT_TL_THRESH,bool); vacuum=~material
print(f"물질 {100*material.mean():.0f}% (t/λ>{MAT_TL_THRESH}) | 기준진공 하위{VAC_PCTL}% | 진공 t/λ noise(MAD)={_vn:.3f} | dark D={tex['dark']:.3g}")
print(f"  ※ 검은 영역이 과하면 MAT_TL_THRESH를 낮추세요(진공 노이즈 {_vn:.3f}의 3~5배 근처가 안전선).")
fig,ax=plt.subplots(1,3,figsize=(13,4.2))
ax[0].imshow(np.log1p(med),cmap="magma"); ax[0].plot(*center,"c+",ms=9); ax[0].set_title("median (log) — amorphous halo"); ax[0].axis("off")
ax[1].imshow((np.clip(mx,0,None)/mx.max())**0.3,cmap="magma"); ax[1].plot(*center,"c+",ms=9); ax[1].set_title("MAX (gamma) — rings/spots"); ax[1].axis("off")
ax[2].imshow(material,cmap="gray"); ax[2].set_title(f"material mask ({100*material.mean():.0f}%) — t/λ>{MAT_TL_THRESH}"); ax[2].axis("off")
plt.tight_layout(); save(fig,"01_overview"); plt.show()

## 1.5) SEI **두께 지표** — t/lambda = ln(I_total / I_transmitted)

두꺼울수록 직접빔 밖으로 산란이 많아짐 → **ln(전체세기/직접빔세기)** 가 상대두께(평균자유행로 단위). dose 무관.
**낮음=얇음(표면/껍데기), 높음=두꺼움.** '표면만 찍었나'를 이 지도·분포로 확인. (절대두께는 lambda 필요; 구분엔 상대값이면 충분)

In [ ]:
# 물질/진공/두께(tmap,tex)는 §1에서 t/lambda로 계산됨 — 여기선 분포·지도만 표시
tin=tmap[material]; tin=tin[np.isfinite(tin)]
tvac=tmap[vacuum]; tvac=tvac[np.isfinite(tvac)]
print(f"dark 자기교정 D={tex['dark']:.3g} | 영점 offset={tex['offset']:.3g} | beam_r={tex['beam_radius']:.1f}px")
print(f"물질 t/lambda(상대두께): mean {np.mean(tin):.2f} | 5~95%: {np.percentile(tin,5):.2f}~{np.percentile(tin,95):.2f}")
print("  낮음=얇음(표면/껍데기), 높음=두꺼움. 분포가 좁고 낮으면 '표면 위주(얇은 shell)'.")
print("  ※ RELATIVE(진공 대비)만 신뢰. 절대 nm은 λ 필요 — 수렴빔·각도분리·무에너지필터라 EELS Malis λ 그대로 못 씀.")
fig,ax=plt.subplots(1,2,figsize=(11,4.2))
im=ax[0].imshow(np.where(material,tmap,np.nan),cmap='viridis'); plt.colorbar(im,ax=ax[0],fraction=0.046)
ax[0].set_title('relative thickness  t/lambda = ln(I_total/I_beam)\n(vacuum-referenced)'); ax[0].axis('off')
ax[1].hist(tin,bins=50,color='0.4',label='material'); ax[1].axvline(0,color='r',ls='--',lw=1,label='vacuum (0)')
ax[1].set_xlabel('t/lambda (relative thickness)'); ax[1].set_ylabel('# positions')
ax[1].set_title('thickness distribution (thin <-> thick)'); ax[1].legend(fontsize=8)
plt.tight_layout(); save(fig,'015_thickness'); plt.show()
save_csv('015_thickness_stats',['metric','value'],
         [['dark',f"{tex['dark']:.4g}"],['offset',f"{tex['offset']:.4g}"],['beam_radius_px',f"{tex['beam_radius']:.1f}"],
          ['material_frac',f"{material.mean():.4f}"],['vac_mean',f"{tvac.mean():.4f}"],['mat_mean',f"{np.mean(tin):.4f}"],
          ['mat_p05',f"{np.percentile(tin,5):.4f}"],['mat_p95',f"{np.percentile(tin,95):.4f}"]])

## 2) [halo] 비정질 FSDP + [ring] 다결정 링 검출

In [ ]:
# --- 이 셀 파라미터 ---
RING_QBEAM = 0.20        # 빔 근처 컷(1/A). 빔이 크면 키우기
RING_QMAX  = 1.0         # 링 검출 상한(1/A)
RING_NSIG  = 1.5         # 링 봉우리 문턱(노이즈 배수). 잡링 많으면 키우기
RING_TOPN  = 8           # 상위 몇 개 링만
# halo(FSDP): 물질 평균의 log-baseline 잔차 봉우리
qd,Id=fds.azimuthal_integrate(fds.average_pattern(cube,material),center,q_per_px=QPP)
halo_q,halo_conf=fds.find_fsdp(qd,Id,q_lo=max(RING_QBEAM,0.10),q_hi=RING_QMAX)
# rings: MAX radial
rings_q=fds.detect_rings(mx,center,QPP,q_beam=RING_QBEAM,q_max=RING_QMAX,nsig=RING_NSIG,top_n=RING_TOPN)
rings_d=[round(1/q,2) for q in rings_q]
print(f"amorphous FSDP: q={halo_q:.3f} d={1/halo_q:.2f}A conf {halo_conf:.1f}")
print(f"rings d(A) = {rings_d}")
qm,Im=fds.azimuthal_integrate(mx,center,q_per_px=QPP)
fig,ax=plt.subplots(1,1,figsize=(9,4))
ax.semilogy(qm,np.clip(Im,1e-2,None),"k-",lw=0.8)
for q in rings_q: ax.axvline(q,color="r",ls=":",lw=0.9)
for c in CANDIDATES:
    for dd,w in fds.COMPOUND_RINGS[c]: ax.axvline(1/dd,color="0.7",ls="--",lw=0.3+0.7*w,alpha=0.35)
ax.axvline(halo_q,color="b",ls="-",lw=1.2)
ax.set_xlabel("q (1/A)"); ax.set_ylabel("MAX I(q)"); ax.set_title(f"rings (red) d={rings_d} | FSDP(blue) d={1/halo_q:.2f}")
plt.tight_layout(); save(fig,"02_rings"); plt.show()

## 3) [spot] Bragg 스팟 검출 (노이즈 배제)

In [ ]:
# --- 이 셀 파라미터 ---
SPOT_NMAD    = 8.0       # 스팟 문턱 = median+NMAD*MAD. ★노이즈 많이 잡히면 키우기(12,15), 약하면 낮추기
SPOT_MINDIST = 3         # 스팟 최소 간격(px)
SPOT_TOPHAT  = 11        # halo 제거 top-hat 크기(px)
SPOT_QMAX    = 1.15      # 스팟 검출 상한(1/A)
spots=fds.detect_spots(mx,center,QPP,q_beam=RING_QBEAM,q_max=SPOT_QMAX,n_mad=SPOT_NMAD,min_dist=SPOT_MINDIST,tophat=SPOT_TOPHAT)
spq=[s[2] for s in spots]
print(f"검출 스팟: {len(spots)}개 (문턱 median+{SPOT_NMAD}*MAD)")
fig,ax=plt.subplots(1,1,figsize=(6,6))
ax.imshow((np.clip(mx,0,None)/mx.max())**0.3,cmap="gray"); ax.plot(*center,"c+",ms=9)
ax.scatter([s[0] for s in spots],[s[1] for s in spots],s=16,facecolors="none",edgecolors="cyan",lw=0.7)
ax.set_title(f"MAX + {len(spots)} spots"); ax.axis("off")
plt.tight_layout(); save(fig,"03_spots"); plt.show()

## 4) [verdict] 상별 판정 (측정 위치 기준 고유성 + 스팟)

In [ ]:
# --- 이 셀 파라미터 ---
MATCH_TOL        = 0.045   # 링/스팟 매칭 허용오차(1/A) = 수렴각 분해능. 데이터 분해능에 맞춰
MIN_UNIQUE_SPOTS = 5       # 이만큼 '고유 위치' 스팟이면 confirmed(약한 링 대신 스팟으로)
phases=fds.score_phases(rings_q,spq,candidates=CANDIDATES,tol=MATCH_TOL,min_unique_spots=MIN_UNIQUE_SPOTS)
ph=list(phases.values())
for e in ph: print(f"  {e.phase:7s} {e.verdict:12s} score {e.score:.2f} | unique {e.unique_d} | spots {e.n_spots}({e.n_unique_spots}u) | missing {e.missing_strong_d}")
fig,ax=plt.subplots(1,1,figsize=(8,3.6))
ax.barh([e.phase for e in ph],[e.score for e in ph],color=[VCOL[e.verdict] for e in ph])
for i,e in enumerate(ph): ax.text(e.score+0.01,i,e.verdict+(f" *{e.unique_d}" if e.unique_d else ""),va="center",fontsize=8)
ax.set_xlim(0,1.3); ax.set_xlabel("ring-match score"); ax.set_title("verdict (green=confirmed, orange=possible, gray=weak)")
plt.tight_layout(); save(fig,"04_verdict"); plt.show()
save_csv("verdicts",["phase","verdict","score","unique_d","missing_strong_d","n_spots","n_unique_spots"],
         [[e.phase,e.verdict,f"{e.score:.3f}","|".join(map(str,e.unique_d)),"|".join(map(str,e.missing_strong_d)),e.n_spots,e.n_unique_spots] for e in ph])

## 4.5) [how-much] 조성 분율 — 링 지문 NNLS (겹겹이 쌓임 = 선형 중첩)

얇은 시료를 통과하면 컬럼에 **쌓인 각 상의 회절이 선형 합**: `I(q) ≈ baseline + h·H(q) + Σ a_p·R_p(q)`.
`R_p`=상 지문(모든 링+강도비), `a_p≥0`=컬럼 양, **`H(q)`=비정질 halo 기저**(FSDP를 흡수해 결정질 오배정 방지 — 실데이터 필수).
**NNLS**로 풀면 분율 `f_p`가 **세기(두께) 무관하게 링 '모양'만으로** 나온다. `crystallinity`=결정질/(결정질+비정질).
**Gram**으로 지문 겹침(분리 난이도) 정직 표시. 두께 상/하위로도 나눠 **두꺼운(더 쌓인) 영역 조성 변화**를 본다.

In [ ]:
# --- 이 셀 파라미터 ---
SIGMA_Q      = 0.035   # 링 지문 폭(1/A) ~ 수렴각 분해능. 링이 넓으면 키우기
BG_WIN_FRAC  = 0.12    # 배경(빔꼬리) 하부포락선 창(프로파일 비율)
DECOMP_QLO   = 0.20    # 분해 하한(1/A, 빔 제외)
DECOMP_QHI   = 1.05    # 분해 상한(1/A)
GROUP_CORR   = 0.90    # Gram |상관|>이 값이면 '분리불가 묶음'으로 병합
HALO_Q       = halo_q  # ★비정질 FSDP 위치(1/A) — §2에서 계산됨. 이걸 halo 기저로 흡수(결정질 오배정 방지)
HALO_SIGMA   = 0.08    # halo 폭(1/A)
def _decomp(msk):
    qq,II=fds.azimuthal_integrate(fds.average_pattern(cube,msk),center,q_per_px=QPP)
    return fds.decompose_fractions(qq,II,candidates=CANDIDATES,sigma_q=SIGMA_Q,bg_win_frac=BG_WIN_FRAC,
             q_lo=DECOMP_QLO,q_hi=DECOMP_QHI,group_corr=GROUP_CORR,halo_q=HALO_Q,halo_sigma=HALO_SIGMA)
# (1) 영역 평균(물질 전체)
dec=_decomp(material)
print("=== 조성 분율 (링 지문 NNLS + 비정질 halo 기저 · 영역평균) ===")
for c in CANDIDATES: print(f"  {c:7s} amount={dec['amounts'][c]:.3g}  frac={dec['fractions'][c]:.3f}")
print(f"  비정질 halo amount={dec['halo_amount']:.3g}  |  결정성(crystallinity)={dec['crystallinity']:.2f}  (낮으면 대부분 비정질)")
print("  [분리불가 묶음(Gram>{:.2f})]:".format(GROUP_CORR))
for k,v in dec['group_fractions'].items():
    if '+' in k: print(f"    {k:26s} {v:.3f}")
print(f"  미설명(residual) peak 비율: {dec['resid_frac']:.2f}  (낮을수록 좋음; 높으면 후보 밖 상/노이즈)")
# (2) 두께 상/하위로 '겹겹이 쌓임'이 조성에 반영되나
tl_med=float(np.nanmedian(tmap[material]))
print(f"  --- 두께 split (t/λ 중앙값 {tl_med:.2f}) ---")
for lab,msk in [("thin(<med) ",material&(tmap<tl_med)),("thick(>=med)",material&(tmap>=tl_med))]:
    if msk.sum()<5: continue
    dd=_decomp(msk)
    print(f"    [{lab}] "+"  ".join(f"{c}:{dd['fractions'][c]:.2f}" for c in CANDIDATES)+f"   crys={dd['crystallinity']:.2f} resid={dd['resid_frac']:.2f}")
# 그림: fit overlay(+halo) | 분율 막대 | Gram heatmap
fig,ax=plt.subplots(1,3,figsize=(15,4.2))
ax[0].plot(dec['q_fit'],dec['peaks'],'k-',lw=1.0,label='measured (peaks)')
ax[0].plot(dec['q_fit'],dec['fit'],'r-',lw=1.1,label='NNLS fit (total)')
ax[0].plot(dec['q_fit'],dec['halo_fit'],'b:',lw=1.0,label='amorphous halo')
for c in CANDIDATES:
    if dec['amounts'][c]>0:
        ax[0].plot(dec['q_fit'],dec['amounts'][c]*fds.phase_ring_profile(dec['q_fit'],c,SIGMA_Q),'--',lw=0.8,label=c)
ax[0].set_xlabel('q (1/A)'); ax[0].set_ylabel('peak I'); ax[0].legend(fontsize=7); ax[0].set_title(f'fingerprint fit (crys={dec["crystallinity"]:.2f}, resid={dec["resid_frac"]:.2f})')
cols=[VCOL.get(next((e.verdict for e in ph if e.phase==c),'weak/absent'),'0.5') for c in CANDIDATES]
ax[1].bar(CANDIDATES,[dec['fractions'][c] for c in CANDIDATES],color=cols)
ax[1].set_ylabel('fraction'); ax[1].set_ylim(0,1); ax[1].set_title('crystalline phase fraction (shape-based)'); ax[1].tick_params(axis='x',rotation=30)
G=dec['gram']; im=ax[2].imshow(G,cmap='magma',vmin=0,vmax=1)
ax[2].set_xticks(range(len(CANDIDATES))); ax[2].set_xticklabels(CANDIDATES,rotation=90,fontsize=7)
ax[2].set_yticks(range(len(CANDIDATES))); ax[2].set_yticklabels(CANDIDATES,fontsize=7)
for i in range(len(CANDIDATES)):
    for j in range(len(CANDIDATES)): ax[2].text(j,i,f"{G[i,j]:.2f}",ha='center',va='center',fontsize=6,color='c')
ax[2].set_title('Gram (fingerprint overlap; high=hard to separate)'); plt.colorbar(im,ax=ax[2],fraction=0.046)
plt.tight_layout(); save(fig,'045_fractions'); plt.show()
save_csv('045_fractions',['phase','amount','fraction'],
         [[c,f"{dec['amounts'][c]:.5g}",f"{dec['fractions'][c]:.4f}"] for c in CANDIDATES]
         +[['_halo',f"{dec['halo_amount']:.5g}",''],['_crystallinity',f"{dec['crystallinity']:.4f}",''],['_resid_frac',f"{dec['resid_frac']:.4f}",'']])
print("주의: 분율=결정질 링 '모양' 기반(두께 무관). FSDP(비정질)는 halo 기저가 흡수 → 상으로 오배정 안 함.")
print("      Gram 높은 쌍은 개별 분율 신뢰↓ → 묶음으로 해석. crystallinity 낮으면 '대부분 비정질'.")

## 5) [where·NBD] 상별 위치 지도 (두께정규화 DF, 진단 링에서)

In [ ]:
# --- 이 셀 파라미터 ---
DQ_RING  = 0.03          # 링 환형 반폭(1/A)
TNORM_IN = 0.20          # 두께정규화 총산란 환형 안쪽(1/A)
TNORM_OUT= 1.0           # 바깥(1/A)
locmaps={}
fig,ax=plt.subplots(1,len(ph),figsize=(3.2*len(ph),3.6))
for a,e in zip(np.atleast_1d(ax),ph):
    d=(e.unique_d[0] if e.unique_d else (e.matched_d[0] if e.matched_d else None))
    if d is None: a.set_title(f"{e.phase}\n(no ring)",fontsize=8); a.axis("off"); locmaps[e.phase]=None; continue
    qc=1/d; m=np.asarray(fds.structural_map(cube,center,(qc-DQ_RING)/QPP,(qc+DQ_RING)/QPP,TNORM_IN/QPP,TNORM_OUT/QPP),float)
    locmaps[e.phase]=m; im=a.imshow(np.where(material,m,np.nan),cmap="inferno"); a.axis("off")
    a.set_title(f"{e.phase} [{e.verdict}]\nd={d:.2f}A amt={np.nanmean(m[material]):.3g}",fontsize=8); plt.colorbar(im,ax=a,fraction=0.046)
fig.suptitle("per-phase NBD location (thickness-normalized DF at diagnostic ring)")
plt.tight_layout(); save(fig,"05_nbd_location"); plt.show()

## 6) [where·cepstral] 상별 특징거리 fluctuation 위치

In [ ]:
# --- 이 셀 파라미터 ---
PHASE_DIST = dict(fds.PHASE_DISTANCE)  # 상별 특징 원자간거리(A). 필요시 값 수정
CEP_HALFWIDTH = 0.35                   # 거리밴드 반폭(A)
cepmaps={}
fig,ax=plt.subplots(1,len(ph),figsize=(3.2*len(ph),3.6))
for a,e in zip(np.atleast_1d(ax),ph):
    d0=PHASE_DIST.get(e.phase)
    try:
        m=np.asarray(fds.fluctuation_image(cube,max(0.4,d0-CEP_HALFWIDTH),d0+CEP_HALFWIDTH,QPP),float)
        mm=m.reshape(scan) if m.ndim==1 else m
        cepmaps[e.phase]=mm
        im=a.imshow(np.where(material,mm,np.nan),cmap="viridis"); a.axis("off")
        a.set_title(f"{e.phase} [{e.verdict}]\ncepstral r~{d0:.2f}A",fontsize=8); plt.colorbar(im,ax=a,fraction=0.046)
    except Exception as ex:
        cepmaps[e.phase]=None
        a.set_title(f"{e.phase}\n(n/a)",fontsize=8); a.axis("off")
fig.suptitle("per-phase cepstral location (fluctuation at diagnostic distance)")
plt.tight_layout(); save(fig,"06_cepstral_location"); plt.show()
print("주의: LiF/Li2O/Li3N은 nn~2.0A 겹쳐 지도 유사(분해능 한계). Li2CO3(1.28)/Li2S(2.47)만 뚜렷 구분.")

## 6.5) [stacking + 검증] 겹겹이 쌓임 — 위치 지도는 '힌트', 지문 분해로 '진짜 있나' 검증

**중요**: §5·§6 위치 지도는 *한 개 링/거리의 신호 세기*로 그려서 — **신호 있음 ≠ 그 상이 진짜 거기 있음**(intensity 착각).
그래서 각 상의 **예상영역(그 상 지도 상위 %)** 을 그 상의 **전체 지문(모든 링+강도비)** 으로 다시 분해해 검증한다:
그 상이 실제로 잡히면 `[OK present]`, 지도만 켜지고 지문이 다른 상을 가리키면 `[X intensity-only]`(착각).
그리고 지도 쌍의 **공간 상관(co-location)** 으로 stacking을 본다 — 단 **Gram 낮은(분리 가능한) 쌍만** 진짜 겹침으로 해석.

In [ ]:
# --- 이 셀 파라미터 ---
STACK_SRC  = "nbd"     # 위치 지도 소스: "nbd"(§5 두께정규화 DF) 또는 "cepstral"(§6)
LOC_PCTL   = 75        # 상별 '예상영역' = 그 상 지도 상위 X% 위치
VALID_FRAC = 0.15      # 검증: 예상영역 지문분해에서 그 상 분율이 이 값 이상이어야 '진짜 존재'
VALID_RESID= 0.5       # 검증: 그리고 residual이 이 값 이하(지문이 실제로 맞음)여야 함
srcmaps = locmaps if STACK_SRC=="nbd" else cepmaps
usable = [e.phase for e in ph if e.verdict!="weak/absent" and srcmaps.get(e.phase) is not None]
M = material
# (B) 상별 예상영역을 '전체 지문'으로 재검증 (intensity 착각 걸러내기) — _decomp 은 §4.5 정의
print(f"=== 상별 예상영역({STACK_SRC} 상위{LOC_PCTL}%) 지문 재검증 ===")
region_rows=[]; validated=[]
for p in usable:
    mp=np.asarray(srcmaps[p],float); thr=np.nanpercentile(mp[M],LOC_PCTL); reg=M&(mp>=thr)
    if reg.sum()<5: continue
    dd=_decomp(reg); fp=dd['fractions'][p]; top=sorted(dd['fractions'].items(),key=lambda kv:-kv[1])[:3]
    ok = (fp>=VALID_FRAC) and (dd['resid_frac']<=VALID_RESID)
    if ok: validated.append(p)
    flag = "[OK present]    " if ok else "[X intensity-only]"
    print(f"  {flag} {p:7s} 영역{int(reg.sum()):4d}px  self={fp:.2f}  top: "+" ".join(f"{k}:{v:.2f}" for k,v in top)+f"  crys={dd['crystallinity']:.2f} resid={dd['resid_frac']:.2f}")
    region_rows.append([p,int(reg.sum()),"present" if ok else "intensity-only"]+[f"{dd['fractions'][c]:.3f}" for c in CANDIDATES]+[f"{dd['crystallinity']:.3f}",f"{dd['resid_frac']:.3f}"])
print(f"  → 지문 검증 통과(진짜 존재): {validated or '없음'}")
# (A) 공간 상관(co-location): 검증 통과한 상들만 (착각 지도 제외)
cusable=[p for p in usable if p in validated]
C=np.corrcoef(np.vstack([np.asarray(srcmaps[p],float)[M].ravel() for p in cusable])) if len(cusable)>=2 else None
# 그림: co-location(검증된 상) | RGB overlay
fig,ax=plt.subplots(1,2,figsize=(11,4.6))
if C is not None:
    im=ax[0].imshow(C,cmap='coolwarm',vmin=-1,vmax=1)
    ax[0].set_xticks(range(len(cusable))); ax[0].set_xticklabels(cusable,rotation=90,fontsize=8)
    ax[0].set_yticks(range(len(cusable))); ax[0].set_yticklabels(cusable,fontsize=8)
    for i in range(len(cusable)):
        for j in range(len(cusable)): ax[0].text(j,i,f"{C[i,j]:.2f}",ha='center',va='center',fontsize=7)
    ax[0].set_title('co-location of VALIDATED phases\n(high=stacked, <0=separated)'); plt.colorbar(im,ax=ax[0],fraction=0.046)
else:
    ax[0].axis('off'); ax[0].set_title(f'co-location: n/a (validated<2: {cusable})')
def _norm(p):
    x=np.asarray(srcmaps[p],float); v=np.where(M,x,np.nan)
    lo,hi=np.nanpercentile(v,5),np.nanpercentile(v,99)
    return np.clip((np.nan_to_num(v)-lo)/(hi-lo+1e-9),0,1)*M
chans=(cusable or usable)[:3]
rgb=np.zeros(M.shape+(3,))
for k,pp in enumerate(chans): rgb[...,k]=_norm(pp)
ax[1].imshow(rgb); ax[1].axis('off')
ax[1].set_title(('overlay R,G,B = '+', '.join(chans)+'\n(blend/white = stacked)') if chans else 'overlay n/a')
plt.tight_layout(); save(fig,'065_stacking'); plt.show()
if region_rows:
    save_csv('065_stacking_regions',['region_phase','n_px','validation']+CANDIDATES+['crystallinity','resid_frac'],region_rows)
print("해석: [X intensity-only]=지도만 켜졌고 지문은 다른 상 → 그 위치에 그 상이 있다고 하면 착각.")
print("      co-location은 '검증 통과' 상들만. Gram 높은 쌍은 지도가 trivial하게 겹치니 Gram 낮은 쌍만 진짜 stacking.")

## 7) 정직한 요약

In [ ]:
conf=[e.phase for e in ph if e.verdict=="confirmed"]; poss=[e.phase for e in ph if e.verdict=="possible"]; weak=[e.phase for e in ph if e.verdict=="weak/absent"]
unexpl=[d for d in rings_d if not any(abs(1/dd-1/d)<=MATCH_TOL for c in CANDIDATES for dd,_ in fds.COMPOUND_RINGS[c])]
print(f"=== {NAME} 요약 ===")
print(f"  확정: {conf or '없음'}   가능: {poss or '없음'}   약함: {weak or '없음'}")
print(f"  미설명 링 d(A): {unexpl}  (후보 5상 밖)")
print("[한계] ~2A 상(LiF/Li2O/Li3N)은 수렴각 분해능으로 겹쳐 특정 제한. cepstral은 분리만, 이름은 링+EDS.")